# Evaluation, tables and figures

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. Requires the outputs of notebooks 01-06. `scripts/paper_numbers.py` then derives the averages, the cost table (Table 5) and every percentage quoted in the paper from the two main-results CSVs.

## 1. Main results tables (Tables 3 and, for the symmetric protocol, the inputs to Table 4)
Run once with `MODE = "asymmetric"` and once with `MODE = "symmetric"`.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================================================
# Build main results table directly from merged output files
#
# Computes, for each model and formulation:
#   - MAE
#   - Delta_cov = |PICP_80 - 0.80|
#   - MPIW_80   = mean(q90 - q10)
#   - QS        = mean pinball loss over q10, q50, q90
#
# Route mapping:
#   - Modular    -> Route1 merged file
#   - Integrated -> Route2 merged file
#
# Works for either:
#   MODE = "symmetric"
#   MODE = "asymmetric"
# ============================================================

import os
import numpy as np
import pandas as pd

# =========================
# 0) Choose experiment
# =========================
MODE = "symmetric"   # change to "asymmetric" when needed

TARGET_COVERAGE = 0.80

# =========================
# 1) File maps
# =========================
if MODE == "symmetric":
    # ---- AFTER ablation Symmetric----
    files_route1 = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route1_point_uq_series_sarimax_symmetric_5min.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_xgb_symmetric_5min.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route1_point_uq_series_bilstm_symmetric_5min.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_bigrulstm_symmetric_5min.csv",
    "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route1_point_uq_series_tft_symmetric_5min.csv",
    "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_zeroshot_symmetric_5min.csv",
    "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_trained_symmetric_5min.csv",

    }

    files_route2 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route2_predictions_prob_sarimax_symmetric_5min.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_xgb_symmetric_5min.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route2_predictions_prob_bilstm_symmetric_5min.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_bigrulstm_symmetric_5min.csv",
        "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route2_predictions_prob_tft_symmetric_5min.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_zeroshot_symmetric_5min.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_trained_symmetric_5min.csv",

    }

elif MODE == "asymmetric":
    files_route1 = {
        "SARIMAX":    OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route1_point_uq_series_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":    OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_xgb_asymmetric_hourly.csv",
        "BiLSTM":     OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM": OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bigrulstm_asymmetric_hourly.csv",
        "TFT":        OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route1_point_uq_series_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":   OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":   OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv",
    }

    files_route2 = {
        "SARIMAX":    OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route2_predictions_prob_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":    OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_xgb_asymmetric_hourly.csv",
        "BiLSTM":     OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM": OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bigrulstm_asymmetric_hourly.csv",
        "TFT":        OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route2_predictions_prob_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":   OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":   OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv",
    }

else:
    raise ValueError("MODE must be 'symmetric' or 'asymmetric'")

OUT_DIR = OUT_ROOT + f"/main_results_table_{MODE}"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 2) Helpers
# =========================
def parse_timestamp(s):
    return pd.to_datetime(s, errors="coerce")

def quantile_pinball_loss(y_true, y_pred, q):
    e = y_true - y_pred
    return np.maximum(q * e, (q - 1.0) * e)

def compute_interval_metrics(actual, q10, q90, target_coverage=0.80):
    inside = (actual >= q10) & (actual <= q90)
    picp = float(np.mean(inside))
    delta_cov = abs(picp - target_coverage)
    mpiw = float(np.mean(q90 - q10))
    return picp, delta_cov, mpiw

def compute_qs(actual, q10, q50, q90):
    # Average pinball loss across q10, q50, q90
    l10 = quantile_pinball_loss(actual, q10, 0.10)
    l50 = quantile_pinball_loss(actual, q50, 0.50)
    l90 = quantile_pinball_loss(actual, q90, 0.90)
    return float(np.mean((l10 + l50 + l90) / 3.0))

def load_route1(path, model_name):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route1/{model_name}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    required = {"timestamp", "actual", "point_pred", "q10", "q50", "q90"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"[Route1/{model_name}] Missing columns {missing}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route1/{model_name}] Invalid timestamps")

    return df[["timestamp", "actual", "point_pred", "q10", "q50", "q90"]].copy()

def load_route2(path, model_name):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route2/{model_name}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Actual Power (W)": "actual",
        "q10 (W)": "q10",
        "q50 (W)": "q50",
        "q90 (W)": "q90",
    }
    df = df.rename(columns=rename_map)

    required = {"timestamp", "actual", "q10", "q50", "q90"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"[Route2/{model_name}] Missing columns {missing}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route2/{model_name}] Invalid timestamps")

    return df[["timestamp", "actual", "q10", "q50", "q90"]].copy()

def summarize_modular(df):
    actual = df["actual"].to_numpy(dtype=float)
    point  = df["point_pred"].to_numpy(dtype=float)
    q10    = df["q10"].to_numpy(dtype=float)
    q50    = df["q50"].to_numpy(dtype=float)
    q90    = df["q90"].to_numpy(dtype=float)

    mae = float(np.mean(np.abs(actual - point)))
    _, delta_cov, mpiw = compute_interval_metrics(actual, q10, q90, TARGET_COVERAGE)
    qs = compute_qs(actual, q10, q50, q90)

    return {
        "MAE": mae,
        "Delta_cov": delta_cov,
        "MPIW_80": mpiw,
        "QS": qs,
    }

def summarize_integrated(df):
    actual = df["actual"].to_numpy(dtype=float)
    q10    = df["q10"].to_numpy(dtype=float)
    q50    = df["q50"].to_numpy(dtype=float)
    q90    = df["q90"].to_numpy(dtype=float)

    mae = float(np.mean(np.abs(actual - q50)))
    _, delta_cov, mpiw = compute_interval_metrics(actual, q10, q90, TARGET_COVERAGE)
    qs = compute_qs(actual, q10, q50, q90)

    return {
        "MAE": mae,
        "Delta_cov": delta_cov,
        "MPIW_80": mpiw,
        "QS": qs,
    }

def bold_best_pair(v1, v2, lower_is_better=True, decimals=2):
    if lower_is_better:
        best = min(v1, v2)
    else:
        best = max(v1, v2)

    s1 = f"{v1:.{decimals}f}"
    s2 = f"{v2:.{decimals}f}"

    if np.isclose(v1, best):
        s1 = r"\textbf{" + s1 + "}"
    if np.isclose(v2, best):
        s2 = r"\textbf{" + s2 + "}"
    return s1, s2

# =========================
# 3) Compute results
# =========================
rows = []

for model in files_route1.keys():
    df_r1 = load_route1(files_route1[model], model)
    df_r2 = load_route2(files_route2[model], model)

    mod = summarize_modular(df_r1)
    integ = summarize_integrated(df_r2)

    rows.append({
        "Model": model,
        "Formulation": "Modular",
        **mod
    })
    rows.append({
        "Model": model,
        "Formulation": "Integrated",
        **integ
    })

results_df = pd.DataFrame(rows)

# Optional fixed ordering
model_order = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM (ZT)", "LLM (FT)"]
results_df["Model"] = pd.Categorical(results_df["Model"], categories=model_order, ordered=True)
results_df["Formulation"] = pd.Categorical(results_df["Formulation"], categories=["Modular", "Integrated"], ordered=True)
results_df = results_df.sort_values(["Model", "Formulation"]).reset_index(drop=True)

# =========================
# 4) Save numeric table
# =========================
csv_path = os.path.join(OUT_DIR, f"main_results_table_{MODE}.csv")
results_df.to_csv(csv_path, index=False)

print("\nNumeric table:")
print(results_df.to_string(index=False))
print(f"\nSaved CSV: {csv_path}")

# =========================
# 5) Print LaTeX rows
# =========================
print("\nLaTeX rows:\n")

for model in model_order:
    pair = results_df[results_df["Model"] == model].reset_index(drop=True)
    if len(pair) != 2:
        continue

    r_mod = pair.iloc[0]
    r_int = pair.iloc[1]

    mae_mod, mae_int = bold_best_pair(r_mod["MAE"],        r_int["MAE"],        lower_is_better=True, decimals=2)
    dc_mod,  dc_int  = bold_best_pair(r_mod["Delta_cov"],  r_int["Delta_cov"],  lower_is_better=True, decimals=3)
    mp_mod,  mp_int  = bold_best_pair(r_mod["MPIW_80"],    r_int["MPIW_80"],    lower_is_better=True, decimals=2)
    qs_mod,  qs_int  = bold_best_pair(r_mod["QS"],         r_int["QS"],         lower_is_better=True, decimals=2)

    print(
        f"{model} & Modular & {mae_mod} & {dc_mod} & {mp_mod} & {qs_mod} \\\\"
    )
    print(
        f" & Integrated & {mae_int} & {dc_int} & {mp_int} & {qs_int} \\\\hline"
    )

# =========================
# 6) Optional full LaTeX table
# =========================
latex_path = os.path.join(OUT_DIR, f"main_results_table_{MODE}.tex")

with open(latex_path, "w") as f:
    f.write("\\begin{table}[!htbp]\n")
    f.write("\\centering\n")
    f.write("\\caption{Performance comparison of modular and integrated uncertainty across model families. Lower is better for MAE, $\\Delta_{cov}$, MPIW$_{80}$, and QS.}\n")
    f.write("\\label{tab:main_results}\n")
    f.write("\\resizebox{\\columnwidth}{!}{%\n")
    f.write("\\begin{tabular}{lccccc}\\hline\n")
    f.write("&  & {Point} & \\multicolumn{2}{c}{{Interval}} & {Distributional} \\\\\\cline{3-6}\n")
    f.write("Model & Formulation & MAE & $\\Delta_{cov}$ & MPIW$_{80}$ & QS \\\\\\hline\n")

    for model in model_order:
        pair = results_df[results_df["Model"] == model].reset_index(drop=True)
        if len(pair) != 2:
            continue

        r_mod = pair.iloc[0]
        r_int = pair.iloc[1]

        mae_mod, mae_int = bold_best_pair(r_mod["MAE"],        r_int["MAE"],        lower_is_better=True, decimals=2)
        dc_mod,  dc_int  = bold_best_pair(r_mod["Delta_cov"],  r_int["Delta_cov"],  lower_is_better=True, decimals=3)
        mp_mod,  mp_int  = bold_best_pair(r_mod["MPIW_80"],    r_int["MPIW_80"],    lower_is_better=True, decimals=2)
        qs_mod,  qs_int  = bold_best_pair(r_mod["QS"],         r_int["QS"],         lower_is_better=True, decimals=2)

        f.write(f"{model} & Modular & {mae_mod} & {dc_mod} & {mp_mod} & {qs_mod} \\\\\n")
        f.write(f" & Integrated & {mae_int} & {dc_int} & {mp_int} & {qs_int} \\\\\\hline\n")

    f.write("\\end{tabular}%\n")
    f.write("}\n")
    f.write("\\end{table}\n")

print(f"\nSaved LaTeX table: {latex_path}")

## 2. Delta table (Table 4): symmetric minus asymmetric

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================================================
# Build delta-results table from symmetric and asymmetric avg tables
#
# Delta definition:
#   DELTA = symmetric - asymmetric
#
# Interpretation:
#   negative delta  -> improvement after ablating asymmetry
#   positive delta  -> worse after ablating asymmetry
#
# Green cells are applied to negative deltas.
# ============================================================

import os
import numpy as np
import pandas as pd

# =========================
# 0) Input files
# =========================
ASYM_CSV = OUT_ROOT + "/main_results_table_asymmetric/main_results_table_asymmetric.csv"
SYM_CSV  = OUT_ROOT + "/main_results_table_symmetric/main_results_table_symmetric.csv"

OUT_DIR = OUT_ROOT + "/delta_results_table"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "delta_results_table.csv")
OUT_TEX = os.path.join(OUT_DIR, "delta_results_table.tex")

# =========================
# 1) Settings
# =========================
MODEL_ORDER = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM (ZT)", "LLM (FT)"]
FORM_ORDER = ["Modular", "Integrated"]

METRICS = ["MAE", "Delta_cov", "MPIW_80", "QS"]

# If your previous CSV used a different column spelling, rename here
RENAME_MAP = {
    "MPIW80": "MPIW_80",
    "MPIW": "MPIW_80",
    "DeltaCov": "Delta_cov",
    "delta_cov": "Delta_cov",
}

# =========================
# 2) Helpers
# =========================
def load_avg_table(path, tag):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{tag}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns=RENAME_MAP)

    required = {"Model", "Formulation", "MAE", "Delta_cov", "MPIW_80", "QS"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"[{tag}] Missing columns: {missing}\nFound: {df.columns.tolist()}")

    df = df[["Model", "Formulation", "MAE", "Delta_cov", "MPIW_80", "QS"]].copy()

    # normalize strings
    df["Model"] = df["Model"].astype(str).str.strip()
    df["Formulation"] = df["Formulation"].astype(str).str.strip()

    # numeric safety
    for c in METRICS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if df[METRICS].isna().any().any():
        bad = df[df[METRICS].isna().any(axis=1)]
        raise ValueError(f"[{tag}] NaNs found in metric columns:\n{bad}")

    return df

def fmt_signed(x, decimals=2):
    if x > 0:
        return f"+{x:.{decimals}f}"
    elif x < 0:
        return f"{x:.{decimals}f}"
    else:
        return f"{x:.{decimals}f}"

def fmt_signed_cov(x):
    # your example shows 3 decimals for Delta_cov
    if x > 0:
        return f"+{x:.3f}"
    elif x < 0:
        return f"{x:.3f}"
    else:
        return f"{x:.3f}"

def green_if_improved(text, value):
    # lower is better for all metrics, so negative delta = improvement
    if value < 0:
        return r"\cellcolor{green!70}" + text
    return text

# =========================
# 3) Load both tables
# =========================
df_asym = load_avg_table(ASYM_CSV, "asymmetric")
df_sym  = load_avg_table(SYM_CSV,  "symmetric")

# enforce ordering
df_asym["Model"] = pd.Categorical(df_asym["Model"], categories=MODEL_ORDER, ordered=True)
df_asym["Formulation"] = pd.Categorical(df_asym["Formulation"], categories=FORM_ORDER, ordered=True)
df_sym["Model"] = pd.Categorical(df_sym["Model"], categories=MODEL_ORDER, ordered=True)
df_sym["Formulation"] = pd.Categorical(df_sym["Formulation"], categories=FORM_ORDER, ordered=True)

df_asym = df_asym.sort_values(["Model", "Formulation"]).reset_index(drop=True)
df_sym  = df_sym.sort_values(["Model", "Formulation"]).reset_index(drop=True)

# =========================
# 4) Merge and compute deltas
# =========================
merged = df_sym.merge(
    df_asym,
    on=["Model", "Formulation"],
    suffixes=("_sym", "_asym"),
    how="inner"
)

expected_rows = len(MODEL_ORDER) * len(FORM_ORDER)
if len(merged) != expected_rows:
    print(f"Warning: merged rows = {len(merged)}, expected = {expected_rows}")

delta_rows = []
for _, r in merged.iterrows():
    row = {
        "Model": r["Model"],
        "Formulation": r["Formulation"],
        "Delta_MAE":       r["MAE_sym"]       - r["MAE_asym"],
        "Delta_Delta_cov": r["Delta_cov_sym"] - r["Delta_cov_asym"],
        "Delta_MPIW_80":   r["MPIW_80_sym"]   - r["MPIW_80_asym"],
        "Delta_QS":        r["QS_sym"]        - r["QS_asym"],
    }
    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows)
delta_df["Model"] = pd.Categorical(delta_df["Model"], categories=MODEL_ORDER, ordered=True)
delta_df["Formulation"] = pd.Categorical(delta_df["Formulation"], categories=FORM_ORDER, ordered=True)
delta_df = delta_df.sort_values(["Model", "Formulation"]).reset_index(drop=True)

delta_df.to_csv(OUT_CSV, index=False)

print("\nDelta numeric table:")
print(delta_df.to_string(index=False))
print(f"\nSaved CSV: {OUT_CSV}")

# =========================
# 5) Print LaTeX rows only
# =========================
print("\nLaTeX rows:\n")

for model in MODEL_ORDER:
    pair = delta_df[delta_df["Model"] == model].reset_index(drop=True)
    if len(pair) != 2:
        continue

    for i, r in pair.iterrows():
        mae_txt = green_if_improved(fmt_signed(r["Delta_MAE"], 2), r["Delta_MAE"])
        dc_txt  = green_if_improved(fmt_signed_cov(r["Delta_Delta_cov"]), r["Delta_Delta_cov"])
        mp_txt  = green_if_improved(fmt_signed(r["Delta_MPIW_80"], 2), r["Delta_MPIW_80"])
        qs_txt  = green_if_improved(fmt_signed(r["Delta_QS"], 2), r["Delta_QS"])

        model_txt = model if i == 0 else ""
        form_txt = r["Formulation"]

        line_end = r"\\"
        if i == 1:
            line_end = r"\\\hline"

        print(
            f"{model_txt} "
            f"& {form_txt} "
            f"& {mae_txt} "
            f"& {dc_txt} "
            f"& {mp_txt} "
            f"& {qs_txt} "
            f"{line_end}"
        )

# =========================
# 6) Save full LaTeX table
# =========================
with open(OUT_TEX, "w") as f:
    f.write("\\begin{table}[!htbp]\n")
    f.write("\\centering\n")
    f.write("\\caption{Effect of ablating the train-deployment asymmetry. Green cells indicate improvement.}\n")
    f.write("\\label{tab:delta_results}\n")
    f.write("\\resizebox{\\columnwidth}{!}{%\n")
    f.write("\\begin{tabular}{lccccc}\\hline\n")
    f.write("Model & Formulation & $\\Delta$MAE & $\\Delta(\\Delta_{cov})$ & $\\Delta$MPIW$_{80}$ & $\\Delta$QS \\\\\\hline\n\n")

    for model in MODEL_ORDER:
        pair = delta_df[delta_df["Model"] == model].reset_index(drop=True)
        if len(pair) != 2:
            continue

        for i, r in pair.iterrows():
            mae_txt = green_if_improved(fmt_signed(r["Delta_MAE"], 2), r["Delta_MAE"])
            dc_txt  = green_if_improved(fmt_signed_cov(r["Delta_Delta_cov"]), r["Delta_Delta_cov"])
            mp_txt  = green_if_improved(fmt_signed(r["Delta_MPIW_80"], 2), r["Delta_MPIW_80"])
            qs_txt  = green_if_improved(fmt_signed(r["Delta_QS"], 2), r["Delta_QS"])

            model_txt = model if i == 0 else ""
            form_txt = r["Formulation"]

            if i == 0:
                f.write(f"{model_txt} \n")
                f.write(f"& {form_txt} \n")
                f.write(f"& {mae_txt} \n")
                f.write(f"& {dc_txt} \n")
                f.write(f"& {mp_txt} \n")
                f.write(f"& {qs_txt} \\\\\n\n")
            else:
                f.write(f"& {form_txt} \n")
                f.write(f"& {mae_txt} \n")
                f.write(f"& {dc_txt} \n")
                f.write(f"& {mp_txt} \n")
                f.write(f"& {qs_txt} \\\\\\hline\n\n")

    f.write("\\end{tabular}%\n")
    f.write("}\n")
    f.write("\\end{table}\n")

print(f"\nSaved LaTeX table: {OUT_TEX}")

## 3. Coverage-width figure (Figure 3)

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================================================
# Side-by-side Coverage–Width Comparison
# Left : Asymmetric
# Right: Symmetric
# Cross-panel arrows connect matching model/formulation points
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch

# =========================
# 0) Setup
# =========================
BACKBONES = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM_ZT", "LLM_FT"]

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

OUT_PATH = OUT_ROOT + "/picp_asym_vs_sym_side_by_side.png"

# =========================
# 1) File maps
# =========================
files_route1_asym = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route1_point_uq_series_sarimax_asymmetric_native_hourly.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_xgb_asymmetric_hourly.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bilstm_asymmetric_hourly.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bigrulstm_asymmetric_hourly.csv",
    "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route1_point_uq_series_tft_asymmetric_hourly_FIXED.csv",
    "LLM_ZT":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv",
    "LLM_FT":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv",
}

files_route2_asym = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route2_predictions_prob_sarimax_asymmetric_native_hourly.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_xgb_asymmetric_hourly.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bilstm_asymmetric_hourly.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bigrulstm_asymmetric_hourly.csv",
    "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route2_predictions_prob_tft_asymmetric_hourly_FIXED.csv",
    "LLM_ZT":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv",
    "LLM_FT":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv",
}

# ---- AFTER ablation Symmetric----
files_route1_sym = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route1_point_uq_series_sarimax_symmetric_5min.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_xgb_symmetric_5min.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route1_point_uq_series_bilstm_symmetric_5min.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_bigrulstm_symmetric_5min.csv",
    "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route1_point_uq_series_tft_symmetric_5min.csv",
    "LLM_ZT":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_zeroshot_symmetric_5min.csv",
    "LLM_FT":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_trained_symmetric_5min.csv",

}

files_route2_sym = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route2_predictions_prob_sarimax_symmetric_5min.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_xgb_symmetric_5min.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route2_predictions_prob_bilstm_symmetric_5min.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_bigrulstm_symmetric_5min.csv",
    "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route2_predictions_prob_tft_symmetric_5min.csv",
    "LLM_ZT":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_zeroshot_symmetric_5min.csv",
    "LLM_FT":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_trained_symmetric_5min.csv",

}

# =========================
# 2) Helpers
# =========================
def parse_timestamp(s):
    return pd.to_datetime(s, errors="coerce")

def add_time_keys(df):
    df = df.copy()
    df["day"] = df["timestamp"].dt.normalize()
    df["hour"] = df["timestamp"].dt.hour
    return df

def load_route1(path, backbone):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    req = {"timestamp", "actual", "point_pred", "q10", "q50", "q90"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"[Route1/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route1/{backbone}] Invalid timestamps in: {path}")

    out = df[["timestamp", "actual", "q10", "q50", "q90"]].copy()
    out["central"] = out["q50"]
    out["route"] = "Route1_UQ"
    out["backbone"] = backbone
    return add_time_keys(out)

def load_route2(path, backbone):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Actual Power (W)": "actual",
        "q10 (W)": "q10",
        "q50 (W)": "q50",
        "q90 (W)": "q90",
    }
    df = df.rename(columns=rename_map)

    req = {"timestamp", "actual", "q10", "q50", "q90"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"[Route2/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route2/{backbone}] Invalid timestamps in: {path}")

    df["central"] = df["q50"]
    df["route"] = "Route2_Prob"
    df["backbone"] = backbone
    df = df[["timestamp", "actual", "q10", "q50", "q90", "central", "route", "backbone"]]
    return add_time_keys(df)

def build_df_all(files_r1, files_r2):
    parts = []
    for bb in BACKBONES:
        p1 = files_r1[bb]
        p2 = files_r2[bb]
        if not os.path.exists(p1):
            raise FileNotFoundError(f"Missing Route1 file for {bb}: {p1}")
        if not os.path.exists(p2):
            raise FileNotFoundError(f"Missing Route2 file for {bb}: {p2}")
        parts.append(load_route1(p1, bb))
        parts.append(load_route2(p2, bb))
    return pd.concat(parts, ignore_index=True).sort_values(["timestamp", "backbone", "route"])

def coverage_width_stats(df):
    inside = (df["actual"] >= df["q10"]) & (df["actual"] <= df["q90"])
    picp = float(inside.mean())
    mpiw = float((df["q90"] - df["q10"]).mean())
    return picp, mpiw

def compute_cw(df_all):
    rows = []
    for bb in BACKBONES:
        for route, formulation in [("Route1_UQ", "Modular U"), ("Route2_Prob", "Integrated U")]:
            cur = df_all[(df_all["backbone"] == bb) & (df_all["route"] == route)].copy()
            if cur.empty:
                continue
            picp, mpiw = coverage_width_stats(cur)
            rows.append({
                "Backbone": bb,
                "Formulation": formulation,
                "PICP": picp,
                "MPIW": mpiw,
            })
    return pd.DataFrame(rows)

# =========================
# 3) Load and summarize
# =========================
df_asym = build_df_all(files_route1_asym, files_route2_asym)
df_sym  = build_df_all(files_route1_sym, files_route2_sym)

cw_asym = compute_cw(df_asym)
cw_sym  = compute_cw(df_sym)

print("Asymmetric summary:")
print(cw_asym.sort_values(["Backbone", "Formulation"]).to_string(index=False))
print("\nSymmetric summary:")
print(cw_sym.sort_values(["Backbone", "Formulation"]).to_string(index=False))

# =========================
# 4) Plot settings (final high-quality)
# =========================
plt.rcParams.update({
    "font.size": 22,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "legend.title_fontsize": 19,
    "figure.dpi": 300,
    "savefig.dpi": 1200,
    "axes.edgecolor": "black",
    "axes.linewidth": 1.8,
})

colors = plt.cm.tab10.colors
color_map = {bb: colors[i % len(colors)] for i, bb in enumerate(BACKBONES)}

marker_map = {
    "Modular U": "^",
    "Integrated U": "s"
}

# =========================
# 5) Limits
# =========================
all_mpiw = np.concatenate([cw_asym["MPIW"], cw_sym["MPIW"]])
all_picp = np.concatenate([cw_asym["PICP"], cw_sym["PICP"]])

x_min, x_max = np.min(all_mpiw), np.max(all_mpiw)
y_min, y_max = np.min(all_picp), np.max(all_picp)

x_pad = 0.03 * (x_max - x_min)
y_pad = 0.04 * (y_max - y_min)

x_lim = (x_min - x_pad, x_max + x_pad)
y_lim = (max(0, y_min - y_pad), min(1, y_max + y_pad))

# =========================
# 6) Figure
# =========================
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(18, 7.2),
    sharex=True,
    sharey=True
)
# TOP LEFT (Modular vs Integrated)
fig.text(
    0.5, 0.97,
    "Modular (post-hoc)        Integrated (in-model)        Asymmetry effect\n"
    "• Closer to ideal         • Under-coverage / wide      → MPIW₈₀ decreases\n"
    "  • Stable PICP             • Large shifts               → intervals inflated\n"
    "       • Moderate width          • Sensitive to asymmetry     → Calibration inconsistent\n"
    "         (Robust)                (Unstable Behavior)                               "    ,
    ha="center",
    va="top",
    fontsize=14,
    family="monospace",   # <-- critical for alignment
    bbox=dict(facecolor="white", edgecolor="black", alpha=0.9)
)

for ax in [ax1, ax2]:
    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)

    # black reference line
    ax.axhline(0.8, linestyle="--", linewidth=2.2, color="black")

    # black grid
    ax.grid(True, linestyle="--", linewidth=1.2, color="black", alpha=0.25)

    ax.set_xlabel("MPIW$_{80}$")

ax1.set_ylabel("PICP")

# =========================
# 6.5) Ideal region
# =========================
for ax in [ax1, ax2]:
    ideal_x = x_lim[0] + 0.35 * (x_lim[1] - x_lim[0])
    ideal_y = 0.67

    ax.fill_betweenx(
        y=[ideal_y, y_lim[1]-.05],
        x1=x_lim[0],
        x2=ideal_x,
        color="black",
        alpha=0.06,
        zorder=0
    )

    ax.text(
        x_lim[0] + 0.015 * (x_lim[1] - x_lim[0]),
        ideal_y + 0.012,
        "Ideal region",
        fontsize=18,
        color="black",
        va="bottom",
        zorder=2
    )
# =========================
# 7) Scatter (larger + sharp)
# =========================
coords_asym = {}
coords_sym = {}

for bb in BACKBONES:
    for formulation in ["Modular U", "Integrated U"]:
        cur = cw_asym[(cw_asym["Backbone"] == bb) &
                      (cw_asym["Formulation"] == formulation)]
        if cur.empty:
            continue

        x, y = float(cur["MPIW"]), float(cur["PICP"])
        coords_asym[(bb, formulation)] = (x, y)

        ax1.scatter(
            x, y,
            s=320,
            marker=marker_map[formulation],
            facecolor=color_map[bb],
            edgecolor="black",
            linewidth=1.3,
            zorder=4
        )

for bb in BACKBONES:
    for formulation in ["Modular U", "Integrated U"]:
        cur = cw_sym[(cw_sym["Backbone"] == bb) &
                     (cw_sym["Formulation"] == formulation)]
        if cur.empty:
            continue

        x, y = float(cur["MPIW"]), float(cur["PICP"])
        coords_sym[(bb, formulation)] = (x, y)

        ax2.scatter(
            x, y,
            s=320,
            marker=marker_map[formulation],
            facecolor=color_map[bb],
            edgecolor="black",
            linewidth=1.3,
            zorder=4
        )

# =========================
# 8) Cross arrows (BLACK)
# =========================
for bb in BACKBONES:
    for formulation in ["Modular U", "Integrated U"]:
        key = (bb, formulation)
        if key not in coords_asym or key not in coords_sym:
            continue

        x1, y1 = coords_asym[key]
        x2, y2 = coords_sym[key]

        con = ConnectionPatch(
            xyA=(x2, y2), coordsA=ax2.transData,
            xyB=(x1, y1), coordsB=ax1.transData,
            arrowstyle="->",
            lw=1.0,
            color="black",
            mutation_scale=22
        )
        fig.add_artist(con)


# =========================
# 9) Legends (clean + black)
# =========================
form_handles = [
    Line2D([0], [0], marker="^", color="black",
           linestyle="None", markersize=14, label="Modular U"),
    Line2D([0], [0], marker="s", color="black",
           linestyle="None", markersize=14, label="Integrated U"),
]

label_map = {
    "LLM_ZT": "LLM (ZT)",
    "LLM_FT": "LLM (FT)"
}

model_handles = [
    Line2D([0], [0], color=color_map[bb],
           lw=4, label=label_map.get(bb, bb))
    for bb in BACKBONES
]

ax2.legend(handles=form_handles, loc="lower center", title="Formulation")
ax2.legend(handles=model_handles, loc="lower right", title="Model")

# =========================
# 10) Subcaptions
# =========================
ax1.text(
    0.5, -0.24,
    "(a) Asymmetry inflates width (MPIW₈₀↑)\n"
    "with inconsistent calibration",
    transform=ax1.transAxes,
    ha="center",
    va="top",
    fontsize=20
)

ax2.text(
    0.5, -0.24,
    "(b) Symmetry reduces width (MPIW₈₀↓)\n"
    "while calibration remains model-dependent",
    transform=ax2.transAxes,
    ha="center",
    va="top",
    fontsize=20
)

# =========================
# 11) Save (CRITICAL)
# =========================
plt.subplots_adjust(top=0.78)

plt.savefig(
    OUT_PATH.replace(".png", ".pdf"),   # vector version (BEST)
    bbox_inches="tight"
)

plt.savefig(
    OUT_PATH,
    dpi=1200,
    bbox_inches="tight"
)

plt.show()

## 4. Forecast waveforms (Figure 4)
Set `MODE` to `"asymmetric"` or `"symmetric"`.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================================================
# Compare model families directly from merged CSV files
#
# Produces:
#   - Back-to-back overlay plots across selected days
#     (Actual + 3 schemes in one figure per backbone)
#     FIXED: consistent colors across both days
#   - Ramp carpets (ΔP) per backbone / route on common days only
#
# Reads directly from saved merged files.
# No dependency on old notebook variables.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0) Choose experiment
# =========================
MODE = "symmetric"   # "symmetric" or "asymmetric"

# Optional manual restriction; otherwise common days are inferred automatically
MANUAL_DAYS_KEEP = None
# Example:
# MANUAL_DAYS_KEEP = ["2024-12-24", "2024-12-30"]

SCHEME_LABEL = {
    "Route1_Point": "Point Prediction",
    "Route1_UQ":    "Modular U",
    "Route2_Prob":  "Integrated U",
}

BACKBONES = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM (ZT)", "LLM (FT)"]

# =========================
# Fixed colors — consistent across both days in back-to-back plots
# =========================
COLOR_ACTUAL    = "#2c7bb6"   # blue
COLOR_POINT     = "#d7191c"   # red
COLOR_MODULAR   = "#fdae61"   # orange
COLOR_INTEGRATED = "#1a9641"  # green

FIG_DIR = f"figs_{MODE}_from_files"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.bbox": "tight"
})

# =========================
# 1) File maps
# =========================
if MODE == "symmetric":
    files_route1 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route1_point_uq_series_sarimax_symmetric_5min.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_xgb_symmetric_5min.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route1_point_uq_series_bilstm_symmetric_5min.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_bigrulstm_symmetric_5min.csv",
        "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route1_point_uq_series_tft_symmetric_5min.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_zeroshot_symmetric_5min.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_trained_symmetric_5min.csv",
    }

    files_route2 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route2_predictions_prob_sarimax_symmetric_5min.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_xgb_symmetric_5min.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route2_predictions_prob_bilstm_symmetric_5min.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_bigrulstm_symmetric_5min.csv",
        "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route2_predictions_prob_tft_symmetric_5min.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_zeroshot_symmetric_5min.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_trained_symmetric_5min.csv",
    }

elif MODE == "asymmetric":
    files_route1 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route1_point_uq_series_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_xgb_asymmetric_hourly.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bigrulstm_asymmetric_hourly.csv",
        "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route1_point_uq_series_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv",
    }

    files_route2 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route2_predictions_prob_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_xgb_asymmetric_hourly.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bigrulstm_asymmetric_hourly.csv",
        "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route2_predictions_prob_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv",
    }

else:
    raise ValueError("MODE must be 'symmetric' or 'asymmetric'")

# =========================
# 2) Helpers
# =========================
def scheme_label(route_name: str) -> str:
    return SCHEME_LABEL.get(route_name, route_name)

def parse_timestamp(series):
    return pd.to_datetime(series, errors="coerce")

def add_time_keys(df):
    out = df.copy()
    out["day"]  = out["timestamp"].dt.normalize()
    out["hour"] = out["timestamp"].dt.hour
    return out

def load_route1(path, backbone):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route1/{backbone}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    required = {"timestamp", "actual", "point_pred", "q10", "q50", "q90"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"[Route1/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        bad = df[df["timestamp"].isna()].head(5)
        raise ValueError(f"[Route1/{backbone}] Unparsable timestamps. Examples:\n{bad}")

    base_cols = ["timestamp", "actual"]

    r1_point = df[base_cols + ["point_pred"]].rename(columns={"point_pred": "central"})
    r1_point["route"]    = "Route1_Point"
    r1_point["backbone"] = backbone

    r1_uq = df[base_cols + ["q10", "q50", "q90"]].copy()
    r1_uq["central"]  = r1_uq["q50"]
    r1_uq["route"]    = "Route1_UQ"
    r1_uq["backbone"] = backbone

    return add_time_keys(pd.concat([r1_point, r1_uq], ignore_index=True))

def load_route2(path, backbone):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route2/{backbone}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Actual Power (W)": "actual",
        "q10 (W)":          "q10",
        "q50 (W)":          "q50",
        "q90 (W)":          "q90",
    }
    df = df.rename(columns=rename_map)

    required = {"timestamp", "actual", "q10", "q50", "q90"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"[Route2/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        bad = df[df["timestamp"].isna()].head(5)
        raise ValueError(f"[Route2/{backbone}] Unparsable timestamps. Examples:\n{bad}")

    df["route"]    = "Route2_Prob"
    df["backbone"] = backbone
    df["central"]  = df["q50"]

    return add_time_keys(
        df[["timestamp", "actual", "q10", "q50", "q90", "central", "route", "backbone"]]
    )

def quantile_order_violations(df):
    needed = {"q10", "q50", "q90"}
    if not needed.issubset(df.columns):
        return 0

    q    = df[["q10", "q50", "q90"]].to_numpy(dtype=float)
    mask = np.isfinite(q).all(axis=1)
    if not mask.any():
        return 0

    q = q[mask]
    return int(((q[:, 0] > q[:, 1]) | (q[:, 1] > q[:, 2])).sum())

def assert_regular_groups(df, expected_steps):
    sizes = df.groupby(["backbone", "route", "day"]).size()
    bad   = sizes[sizes != expected_steps]
    if not bad.empty:
        raise AssertionError(f"Unexpected group sizes found:\n{bad}")

def actual_series(df_all):
    return (
        df_all[["timestamp", "actual"]]
        .drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .assign(
            day  = lambda x: x["timestamp"].dt.normalize(),
            hour = lambda x: x["timestamp"].dt.hour
        )
    )

def get_file_days(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "timestamp" not in df.columns:
        return set()
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    return set(ts.dt.normalize().dropna())

def infer_common_days(files_route1, files_route2):
    all_sets = []

    for path in files_route1.values():
        if os.path.exists(path):
            all_sets.append(get_file_days(path))

    for path in files_route2.values():
        if os.path.exists(path):
            all_sets.append(get_file_days(path))

    if not all_sets:
        raise RuntimeError("Could not compute day intersection. No readable files found.")

    return pd.DatetimeIndex(sorted(set.intersection(*all_sets)))

# =========================
# 3) Load everything directly from files
# =========================
parts  = []
loaded = []

for backbone in BACKBONES:
    p1 = files_route1[backbone]
    p2 = files_route2[backbone]

    parts.append(load_route1(p1, backbone))
    loaded.append((backbone, "Route1", p1))

    parts.append(load_route2(p2, backbone))
    loaded.append((backbone, "Route2", p2))

df_all = (
    pd.concat(parts, ignore_index=True)
    .sort_values(["timestamp", "backbone", "route"])
    .reset_index(drop=True)
)

print("\nLoaded files:")
for backbone, route_name, path in loaded:
    print(f"  - {backbone:10s} {route_name}: {path}")

print("\nLoaded rows:", len(df_all))
print("Backbones present:", sorted(df_all["backbone"].unique().tolist()))
print("Routes present:",    sorted(df_all["route"].unique().tolist()))

tmp_freq = (
    df_all[df_all["route"] == "Route2_Prob"]
    .groupby(["backbone", "day"])["timestamp"]
    .nunique()
)
expected_steps = int(tmp_freq.median())
assert_regular_groups(df_all, expected_steps)

viol = quantile_order_violations(df_all[df_all["route"] != "Route1_Point"])
print(f"Quantile ordering violations (q10<=q50<=q90): {viol}")
print(f"Expected steps/day inferred: {expected_steps}")

# =========================
# 3.1) Common days
# =========================
if MANUAL_DAYS_KEEP is None:
    DAYS_KEEP = infer_common_days(files_route1, files_route2)
else:
    DAYS_KEEP = pd.DatetimeIndex(pd.to_datetime(MANUAL_DAYS_KEEP).normalize())

print("Using DAYS_KEEP:", [d.strftime("%Y-%m-%d") for d in DAYS_KEEP])

# =========================
# 4) Plotting utilities
# =========================
def plot_days_back_to_back(df_all, backbone, days, save=True):
    """
    Plot all days side-by-side in a single figure.
    FIXED: explicit color= on every plt.plot / fill_between call so
    colors are identical across Day 1 and Day 2 (and any further days).
    """
    days = pd.DatetimeIndex(pd.to_datetime(days).normalize()).sort_values()

    A   = actual_series(df_all)
    cur = df_all[df_all["backbone"] == backbone].copy()

    if cur.empty:
        print(f"(skip) No data for {backbone}")
        return

    plt.figure(figsize=(11, 4.3))
    plt.title(f"{backbone} - {MODE.capitalize()} - Back-to-Back Days")
    plt.ylabel("Power (W)")

    x_offset    = 0
    tick_positions = []
    tick_labels    = []
    used_labels    = set()

    for day_ts in days:
        A_day = A[A["day"] == day_ts].sort_values("timestamp")
        D_day = cur[cur["day"] == day_ts].sort_values("timestamp")

        if A_day.empty and D_day.empty:
            continue

        r1p = D_day[D_day["route"] == "Route1_Point"].copy()
        r1u = D_day[D_day["route"] == "Route1_UQ"].copy()
        r2p = D_day[D_day["route"] == "Route2_Prob"].copy()

        n_day = max(
            len(A_day) if not A_day.empty else 0,
            len(r1p)   if not r1p.empty   else 0,
            len(r1u)   if not r1u.empty   else 0,
            len(r2p)   if not r2p.empty   else 0,
        )

        if n_day == 0:
            continue

        # ── Actual ──────────────────────────────────────────────────────────
        if not A_day.empty:
            xA    = np.arange(len(A_day)) + x_offset
            label = "Actual" if "Actual" not in used_labels else None
            plt.plot(
                xA,
                A_day["actual"].to_numpy(dtype=float),
                linewidth=2,
                color=COLOR_ACTUAL,       # FIXED: always blue
                label=label,
            )
            used_labels.add("Actual")

        # ── Route1 Point ─────────────────────────────────────────────────────
        if not r1p.empty:
            x1    = np.arange(len(r1p)) + x_offset
            label = "Point Prediction" if "Point Prediction" not in used_labels else None
            plt.plot(
                x1,
                r1p["central"].to_numpy(dtype=float),
                linestyle="--",
                linewidth=1.8,
                color=COLOR_POINT,        # FIXED: always red
                label=label,
            )
            used_labels.add("Point Prediction")

        # ── Route1 UQ (Modular) ──────────────────────────────────────────────
        if not r1u.empty:
            xu         = np.arange(len(r1u)) + x_offset
            band_label = "Modular U: 10-90%" if "Modular U: 10-90%" not in used_labels else None
            mid_label  = "Modular U (q50)"   if "Modular U (q50)"   not in used_labels else None

            plt.fill_between(
                xu,
                r1u["q10"].to_numpy(dtype=float),
                r1u["q90"].to_numpy(dtype=float),
                alpha=0.18,
                color=COLOR_MODULAR,      # FIXED: always orange
                label=band_label,
            )
            plt.plot(
                xu,
                r1u["q50"].to_numpy(dtype=float),
                linewidth=1.3,
                color=COLOR_MODULAR,      # FIXED: always orange
                label=mid_label,
            )
            used_labels.add("Modular U: 10-90%")
            used_labels.add("Modular U (q50)")

        # ── Route2 Prob (Integrated) ─────────────────────────────────────────
        if not r2p.empty:
            x2         = np.arange(len(r2p)) + x_offset
            band_label = "Integrated U: 10-90%" if "Integrated U: 10-90%" not in used_labels else None
            mid_label  = "Integrated U (q50)"   if "Integrated U (q50)"   not in used_labels else None

            plt.fill_between(
                x2,
                r2p["q10"].to_numpy(dtype=float),
                r2p["q90"].to_numpy(dtype=float),
                alpha=0.18,
                color=COLOR_INTEGRATED,   # FIXED: always green
                label=band_label,
            )
            plt.plot(
                x2,
                r2p["q50"].to_numpy(dtype=float),
                linewidth=1.3,
                color=COLOR_INTEGRATED,   # FIXED: always green
                label=mid_label,
            )
            used_labels.add("Integrated U: 10-90%")
            used_labels.add("Integrated U (q50)")

        # Vertical separator between days
        if x_offset > 0:
            plt.axvline(x=x_offset - 0.5, linestyle="--", alpha=0.5)

        tick_positions.append(x_offset + (n_day - 1) / 2)
        tick_labels.append(day_ts.strftime("%b %d"))
        x_offset += n_day

    plt.xticks(tick_positions, tick_labels)
    plt.xlabel("Days")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(ncol=2, frameon=False, loc="upper left")

    if save:
        day_tag = "_to_".join(pd.Timestamp(d).strftime("%Y%m%d") for d in days)
        fname   = os.path.join(
            FIG_DIR,
            f"{MODE}_days_back_to_back_{backbone.replace('-', '').lower()}_{day_tag}.jpg"
        )
        plt.savefig(fname, dpi=300)

    plt.show()


def ramp_carpet(series_2d, title, ylabels, savepath=None):
    dP = np.diff(series_2d, axis=1, prepend=series_2d[:, [0]])

    plt.figure(figsize=(6.5, 3.2))
    plt.title(title)
    plt.xlabel("Time Step Within Day")
    plt.ylabel("Day")

    im = plt.imshow(dP, aspect="auto", origin="upper")
    plt.colorbar(im, label="ΔP (W)")

    if expected_steps == 24:
        xticks = list(range(0, 24, 2))
        plt.xticks(xticks, [str(t) for t in xticks])
        plt.xlabel("Hour of Day")
    elif expected_steps == 288:
        xticks      = list(range(0, 288, 24))
        xticklabels = [str(int(t / 12)) for t in xticks]
        plt.xticks(xticks, xticklabels)
        plt.xlabel("Hour of Day")

    plt.yticks(range(len(ylabels)), ylabels, rotation=90, ha="center", va="center")

    if savepath is not None:
        plt.savefig(savepath, dpi=300)

    plt.show()


def plot_ramp_carpet(df_all, route, backbone, save=True, days_keep=None):
    cur = df_all[
        (df_all["route"] == route) & (df_all["backbone"] == backbone)
    ].copy()

    if days_keep is not None:
        keep = pd.DatetimeIndex(pd.to_datetime(days_keep).normalize())
        cur  = cur[pd.to_datetime(cur["day"]).isin(keep)]

    if cur.empty:
        print(f"(skip) No data for {route} / {backbone} after filtering")
        return

    cur        = cur.sort_values("timestamp").copy()
    cur["step"] = cur.groupby("day").cumcount()

    mat = (
        cur.pivot(index="day", columns="step", values="central")
           .sort_index()
           .reindex(columns=range(expected_steps))
    )

    ylabels  = [pd.Timestamp(d).strftime("%b %d, %Y") for d in mat.index]
    savepath = None
    if save:
        savepath = os.path.join(
            FIG_DIR,
            f"{MODE}_ramp_{backbone.replace('-', '')}_{route}.jpg"
        )

    ramp_carpet(
        mat.to_numpy(dtype=float),
        title    = f"ΔP - {MODE.capitalize()} - {backbone} / {scheme_label(route)}",
        ylabels  = ylabels,
        savepath = savepath,
    )

# =========================
# 5) Make figures
# =========================
for backbone in BACKBONES:
    plot_days_back_to_back(df_all, backbone, DAYS_KEEP, save=True)

for backbone in BACKBONES:
    for route_name in ["Route1_Point", "Route1_UQ", "Route2_Prob"]:
        plot_ramp_carpet(df_all, route_name, backbone=backbone,
                         save=True, days_keep=DAYS_KEEP)

print(f"\nDone. Figures saved in: {FIG_DIR}")

## 5. Quantile reliability (supplementary)
Set `MODE` to `"asymmetric"` or `"symmetric"`.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# =========================
# Combined Quantile Reliability from Files Directly
# Clean legend version
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.lines as mlines
import matplotlib.pyplot as plt

# =====================================================
# 0) Choose experiment
# =====================================================
MODE = "symmetric"   # change to "asymmetric" if needed

# Keep only specific days if desired, else set to None
DAYS_KEEP = None
# Example:
# DAYS_KEEP = ["2024-12-24", "2024-12-30"]

QUANTILES = [0.1, 0.5, 0.9]
BACKBONES = ["SARIMAX", "XGBoost", "BiLSTM", "BiGRU-LSTM", "TFT", "LLM (ZT)", "LLM (FT)"]

# =====================================================
# 1) File maps
# =====================================================
if MODE == "symmetric":
    # ---- AFTER ablation Symmetric----
    files_route1 = {
    "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route1_point_uq_series_sarimax_symmetric_5min.csv",
    "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_xgb_symmetric_5min.csv",
    "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route1_point_uq_series_bilstm_symmetric_5min.csv",
    "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route1_point_uq_series_bigrulstm_symmetric_5min.csv",
    "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route1_point_uq_series_tft_symmetric_5min.csv",
    "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_zeroshot_symmetric_5min.csv",
    "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route1_point_uq_series_llm_trained_symmetric_5min.csv",

    }

    files_route2 = {
            "SARIMAX":     OUT_ROOT + "/sarimax_out_symmetric_5min_route1_standardized_recursive/merged_route2_predictions_prob_sarimax_symmetric_5min.csv",
            "XGBoost":     OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_xgb_symmetric_5min.csv",
            "BiLSTM":      OUT_ROOT + "/bilstm_out_symmetric_5min_fixed/merged_route2_predictions_prob_bilstm_symmetric_5min.csv",
            "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_symmetric_5min_route1_standardized/merged_route2_predictions_prob_bigrulstm_symmetric_5min.csv",
            "TFT":         OUT_ROOT + "/tft_out_symmetric_5min_recursive/merged_route2_predictions_prob_tft_symmetric_5min.csv",
            "LLM (ZT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_zeroshot_symmetric_5min.csv",
            "LLM (FT)":      OUT_ROOT + "/llm_out_symmetric_5min_fixed1/merged_route2_predictions_prob_llm_trained_symmetric_5min.csv",

      }

elif MODE == "asymmetric":
    files_route1 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route1_point_uq_series_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_xgb_asymmetric_hourly.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route1_point_uq_series_bigrulstm_asymmetric_hourly.csv",
        "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route1_point_uq_series_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route1_point_uq_series_llm_trained_asymmetric_hourly.csv",
    }

    files_route2 = {
        "SARIMAX":     OUT_ROOT + "/sarimax_out_asymmetric_NATIVE_hourly/merged_route2_predictions_prob_sarimax_asymmetric_native_hourly.csv",
        "XGBoost":     OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_xgb_asymmetric_hourly.csv",
        "BiLSTM":      OUT_ROOT + "/bilstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bilstm_asymmetric_hourly.csv",
        "BiGRU-LSTM":  OUT_ROOT + "/bigrulstm_out_asymmetric_train5min_testhourly/merged_route2_predictions_prob_bigrulstm_asymmetric_hourly.csv",
        "TFT":         OUT_ROOT + "/tft_out_asymmetric_FIXED_hourly/merged_route2_predictions_prob_tft_asymmetric_hourly_FIXED.csv",
        "LLM (ZT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_zeroshot_asymmetric_hourly.csv",
        "LLM (FT)":      OUT_ROOT + "/llm_out_asymmetric_hourly/merged_route2_predictions_prob_llm_trained_asymmetric_hourly.csv",
    }

else:
    raise ValueError("MODE must be 'symmetric' or 'asymmetric'")

# =====================================================
# 2) Helpers
# =====================================================
def parse_timestamp(s):
    return pd.to_datetime(s, errors="coerce")

def maybe_filter_days(df, days_keep=None):
    if days_keep is None:
        return df.copy()
    keep = pd.to_datetime(list(days_keep)).normalize()
    return df[df["day"].isin(keep)].copy()

def add_time_keys(df):
    df = df.copy()
    df["day"] = df["timestamp"].dt.normalize()
    df["hour"] = df["timestamp"].dt.hour
    return df

def load_route1(path, backbone):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route1/{backbone}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    req = {"timestamp", "actual", "point_pred", "q10", "q50", "q90"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"[Route1/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route1/{backbone}] Invalid timestamps in: {path}")

    out = df[["timestamp", "actual", "q10", "q50", "q90"]].copy()
    out["route"] = "Route1_UQ"
    out["backbone"] = backbone
    return add_time_keys(out)

def load_route2(path, backbone):
    if not os.path.exists(path):
        raise FileNotFoundError(f"[Route2/{backbone}] Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Actual Power (W)": "actual",
        "q10 (W)": "q10",
        "q50 (W)": "q50",
        "q90 (W)": "q90",
    }
    df = df.rename(columns=rename_map)

    req = {"timestamp", "actual", "q10", "q50", "q90"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"[Route2/{backbone}] Missing columns {missing} in: {path}")

    df["timestamp"] = parse_timestamp(df["timestamp"])
    if df["timestamp"].isna().any():
        raise ValueError(f"[Route2/{backbone}] Invalid timestamps in: {path}")

    out = df[["timestamp", "actual", "q10", "q50", "q90"]].copy()
    out["route"] = "Route2_Prob"
    out["backbone"] = backbone
    return add_time_keys(out)

def build_df_all(files_r1, files_r2):
    parts = []
    for bb in BACKBONES:
        parts.append(load_route1(files_r1[bb], bb))
        parts.append(load_route2(files_r2[bb], bb))
    return pd.concat(parts, ignore_index=True).sort_values(["timestamp", "backbone", "route"]).reset_index(drop=True)

# =====================================================
# 3) Build dataframe directly from files
# =====================================================
df_all = build_df_all(files_route1, files_route2)
df_rel = maybe_filter_days(df_all, DAYS_KEEP)

print(f"Mode: {MODE}")
print(f"Rows loaded: {len(df_all)}")
print(f"Rows used for reliability: {len(df_rel)}")

# =====================================================
# 4) Plot settings
# =====================================================
route_name_map = {
    "Route1_UQ": "Modular U",
    "Route2_Prob": "Integrated U"
}

colors = plt.cm.tab10.colors
color_map = {bb: colors[i % len(colors)] for i, bb in enumerate(BACKBONES)}

style_map = {
    "Modular U": {"linestyle": "-",  "marker": "o"},
    "Integrated U": {"linestyle": "--", "marker": "s"}
}

display_name = {
    "SARIMAX": "SARIMAX",
    "XGBoost": "XGBoost",
    "BiLSTM": "BiLSTM",
    "BiGRU-LSTM": "BiGRU-LSTM",
    "TFT": "TFT",
    "LLM (ZT)": "LLM (ZT)",
    "LLM (FT)": "LLM (FT)",
}

# =====================================================
# 5) Plot
# =====================================================
plt.figure(figsize=(7, 5))

for bb in BACKBONES:
    for route, formulation in route_name_map.items():
        cur = df_rel[(df_rel["backbone"] == bb) & (df_rel["route"] == route)].copy()
        if cur.empty:
            continue

        emp = []
        for q in QUANTILES:
            qcol = f"q{int(q * 100)}"
            if qcol not in cur.columns:
                raise ValueError(f"Column {qcol} not found for {bb} / {route}")
            emp_cov = float((cur["actual"] <= cur[qcol]).mean())
            emp.append(emp_cov)

        plt.plot(
            QUANTILES,
            emp,
            color=color_map[bb],
            linestyle=style_map[formulation]["linestyle"],
            marker=style_map[formulation]["marker"],
            linewidth=2,
            markersize=7
        )

# ideal diagonal
plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1.5)

title_txt = f"{MODE.capitalize()} Quantile Reliability"
if DAYS_KEEP is not None:
    title_txt += " (Selected Days)"

plt.title(title_txt, fontsize=14)
plt.xlabel("Nominal Quantile", fontsize=14)
plt.ylabel("Empirical Coverage", fontsize=14)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.tick_params(axis="both", labelsize=13)
plt.grid(True, linestyle="--", alpha=0.5)

# -------------------------
# Legend 1: Backbone (color)
# -------------------------
backbone_handles = [
    mlines.Line2D([], [], color=color_map[bb], linestyle='-', label=display_name[bb])
    for bb in BACKBONES
]

legend1 = plt.legend(
    handles=backbone_handles,
    frameon=False,
    fontsize=12,
    loc="best"
)
plt.gca().add_artist(legend1)

# -------------------------
# Legend 2: Formulation (style)
# -------------------------
formulation_handles = [
    mlines.Line2D(
        [], [],
        color='black',
        linestyle=style_map[f]["linestyle"],
        marker=style_map[f]["marker"],
        label=f
    )
    for f in style_map
]

plt.legend(
    handles=formulation_handles,
    frameon=False,
    fontsize=12,
    loc="lower right"
)

plt.tight_layout()
plt.show()